In [3]:
import pandas as pd
import numpy as np

DATA = "../data"

handle = pd.read_csv(f"{DATA}/state_monthly_handle_raw.csv")
handle["month"] = pd.to_datetime(handle["month"], format="%Y-%m")
handle["log_handle"] = np.log(handle["handle"].clip(lower=1)) 

events = pd.read_csv(f"{DATA}/endorsements.csv")
events["announced_date"] = pd.to_datetime(events["announced_date"])
events["announced_month"] = events["announced_date"].values.astype("datetime64[M]")
events["event_id"] = events["athlete"] + " x " + events["platform"] + " (" + events["announced_date"].dt.strftime("%Y-%m") + ")"

ops = pd.read_csv(f"{DATA}/platform_state_operations.csv")
ops["start_date"] = pd.to_datetime(ops["start_date"])
ops["end_date"] = pd.to_datetime(ops["end_date"])

rows = []
skipped = []

for _, ev in events.iterrows():
    plat_ops = ops[ops["platform"] == ev["platform"]]
    window_start = ev["announced_month"] - pd.DateOffset(months=6)
    window_end = ev["announced_month"] + pd.DateOffset(months=12)

    for _, op in plat_ops.iterrows():
        state = op["state"]
        if pd.isna(op["start_date"]):
            skipped.append((ev["event_id"], state, "platform never operated in this state during study window"))
            continue
        live_at_announcement = op["start_date"] <= ev["announced_month"]
        if not live_at_announcement:
            skipped.append((ev["event_id"], state, f"platform launched in state ({op['start_date'].date()}) after the endorsement announcement -- excluded"))
            continue

        state_handle = handle[handle["state"] == state].copy()
        state_handle = state_handle[(state_handle["month"] >= window_start) & (state_handle["month"] <= window_end)]
        if pd.notna(op["end_date"]):
            state_handle = state_handle[state_handle["month"] <= op["end_date"]]

        for _, r in state_handle.iterrows():
            k = (r["month"].year - ev["announced_month"].year) * 12 + (r["month"].month - ev["announced_month"].month)
            rows.append({
                "event_id": ev["event_id"],
                "athlete": ev["athlete"],
                "platform": ev["platform"],
                "timed_to_major_event": ev["timed_to_major_event"] != "No",
                "state": state,
                "state_event": f"{state}__{ev['event_id']}",
                "month": r["month"],
                "k": k,
                "handle": r["handle"],
                "log_handle": r["log_handle"],
            })

panel = pd.DataFrame(rows)
panel = panel.sort_values(["event_id", "state", "month"]).reset_index(drop=True)
panel.to_csv(f"{DATA}/stacked_event_panel.csv", index=False)

print(f"Stacked event panel: {len(panel)} (event, state, month) observations")
print(f"Events included: {panel['event_id'].nunique()}")
print(f"Distinct treated states: {sorted(panel['state'].unique())}")

Stacked event panel: 224 (event, state, month) observations
Events included: 6
Distinct treated states: ['AZ', 'CO', 'CT']
